In [38]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [39]:
from dotenv import load_dotenv

load_dotenv()

# Create an API client
from anthropic import Anthropic

client = Anthropic()
#model = "claude-sonnet-4-6"
model = "claude-haiku-4-5-20251001"


In [40]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [41]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [42]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [43]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [44]:
# def grade_by_model(test_case, output):
#     # Create evaluation prompt
#     eval_prompt = f"""
#     You are an expert code reviewer. Evaluate this AI-generated solution.
    
#     Task: {test_case["task"]}
#     Solution: {output}
    
#     Provide your evaluation as a structured JSON object with:
#     - "strengths": An array of 1-3 key strengths
#     - "weaknesses": An array of 1-3 key areas for improvement  
#     - "reasoning": A concise explanation of your assessment
#     - "score": A number between 1-10
#     """
    
#     messages = []
#     add_user_message(messages, eval_prompt)
#     add_assistant_message(messages, "```json")
#     eval_text = chat(messages, stop_sequences=["```"])
#     return json.loads(eval_text)

def _fix_invalid_json_escapes(text):
    """Doubles up backslashes that aren't part of a valid JSON escape sequence.
    Needed because the model sometimes quotes regex/code containing raw
    backslashes (e.g. \\d, \\-) directly inside a JSON string value."""
    return re.sub(r'\\(?!["\\/bfnrtu])', r"\\\\", text)

def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.

    Task: {test_case["task"]}
    Solution: {output}

    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10

    If you need to reference code, regex, or file paths in your evaluation, escape every backslash as \\\\ so the result is valid JSON.
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    try:
        return json.loads(eval_text)
    except json.JSONDecodeError:
        return json.loads(_fix_invalid_json_escapes(eval_text))
    

In [ ]:
import ast
import re

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    
def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [46]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # Todo - Grading the result
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [47]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run test case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score across all test cases: {average_score:.2f}")
    
    return results

In [48]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

result = run_eval(dataset)    

Average score across all test cases: 7.00


In [49]:
print(json.dumps(result, indent=2))

[
  {
    "output": "# AWS Lambda Function Configuration (JSON)\n\n```json\n{\n  \"FunctionName\": \"s3-event-processor\",\n  \"Runtime\": \"python3.11\",\n  \"Role\": \"arn:aws:iam::ACCOUNT_ID:role/lambda-execution-role\",\n  \"Handler\": \"index.lambda_handler\",\n  \"Timeout\": 60,\n  \"MemorySize\": 512,\n  \"Description\": \"Lambda function triggered by S3 bucket events\",\n  \"Environment\": {\n    \"Variables\": {\n      \"ENVIRONMENT\": \"production\"\n    }\n  },\n  \"EventInvokeConfig\": {\n    \"MaximumEventAge\": 3600,\n    \"MaximumRetryAttempts\": 2,\n    \"DestinationConfig\": {\n      \"OnFailure\": {\n        \"Type\": \"SQS\",\n        \"Destination\": \"arn:aws:sqs:us-east-1:ACCOUNT_ID:lambda-dlq\"\n      }\n    }\n  }\n}\n```\n\n## S3 Event Trigger Configuration\n\n```json\n{\n  \"EventSourceMappings\": [\n    {\n      \"EventSource\": \"s3:ObjectCreated:*\",\n      \"Filter\": {\n        \"Key\": {\n          \"FilterRules\": [\n            {\n              \"Name\